# 🧪 Thực Nghiệm Nâng Cao RS-LiDAR: Guidance Scale, Particle Scaling & Phase 1 Lookahead Horizon (Kaggle 2x T4)
### **Khung Thực Nghiệm Độc Lập**: Kiểm Chứng Toàn Diện Ba Trụ Cột Động Học Manifold

Notebook này hiện thực hóa **3 Bản Đề Xuất Thực Nghiệm Khoa Học** nhằm chứng minh các ưu thế vượt trội của **RS-LiDAR** so với **LiDAR gốc (ICML 2026)**:
1. **Thực Nghiệm 1 (Guidance Scale Stress-Test: $s \in \{7.5, 12.5, 15.0, 17.5, 20.0\}$)**: 
   - Kiểm chứng giả thuyết: Gradient LiDAR gốc bị rung giật (Test 3). Khi tăng scale $s \ge 17.5$, lực giật bị khuếch đại làm vỡ đa tạp, ảnh bị hỏng.
   - RS-LiDAR nhờ chặn Lipschitz chặt chẽ ($L_\sigma \le \frac{\lambda}{\sigma \sqrt{2\pi}}$) có **Ngưỡng Chịu Lực (Stability Margin)** rộng hơn hẳn, duy trì ảnh sắc nét và đạt đỉnh ImageReward mới ở $s = 17.5 \sim 20.0$.
2. **Thực Nghiệm 2 (Particle Scaling & Ultra-Low Budget: $N \in \{3, 5, 10, 20, 50, 100\}$)**:
   - Kiểm chứng giả thuyết: LiDAR gốc bị sụp đổ Softmax về 1 hạt (Test 2). Ở $N$ thấp ($N=3, 5$) dễ bốc trúng hạt có điểm thưởng giả; ở $N=100$ bị bão hòa.
   - RS-LiDAR kích hoạt sức mạnh tập hợp đa hạt (Multi-particle Consensus), vượt trội ở ngân sách siêu thấp và tiếp tục bứt phá tại $N=100$.
3. **Thực Nghiệm 3 (Phase 1 Lookahead Horizon & Solver Error Truncation: $K \in \{2, 3, 5, 8\}$ DPM steps)**:
   - Kiểm chứng giả thuyết: Khi giảm bước sinh ở Phase 1 để tiết kiệm tính toán (DPM 2 hoặc 3 bước), ảnh sơ khai có nhiều sai số xấp xỉ tần số cao. Bộ reward gốc chấm điểm sai lệch nghiêm trọng.
   - RS-LiDAR thêm nhiễu làm mịn trước khi tính reward giúp **hấp thụ và triệt tiêu sai số của solver**, duy trì hướng lái chính xác ngay cả khi Phase 1 chỉ chạy 2–3 bước DPM (tiết kiệm 60% compute Phase 1).

In [ ]:
import os, sys, torch

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA khả dụng: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    print(f"🎮 Số lượng GPU phát hiện: {n_gpus}")
    for i in range(n_gpus):
        vram = torch.cuda.get_device_properties(i).total_memory / (1024**3)
        print(f"  • GPU {i}: {torch.cuda.get_device_name(i)} ({vram:.2f} GB VRAM)")
else:
    raise RuntimeError("❌ Không phát hiện GPU CUDA! Vui lòng bật GPU trong Runtime Settings.")

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
!nvidia-smi


## 2. Cấu Hình Tham Số Thực Nghiệm Tập Trung
Mặc định chạy trên **20 prompts đại diện** để có biểu đồ so sánh ngay trong ~15–30 phút. Bạn có thể chọn bất kỳ chế độ nào trong 3 thực nghiệm. Mặc định là `1_GUIDANCE_SCALE_SWEEP` vì có thể tái sử dụng ngay $N=50$ hạt Phase 1 đã chạy từ trước, hoàn tất rất nhanh!

In [ ]:
# ==============================================================================
# ⚙️ KHU VỰC CẤU HÌNH TẬP TRUNG (CHỌN CHẾ ĐỘ THỰC NGHIỆM TẠI ĐÂY)
# ==============================================================================
# Chọn chế độ thực nghiệm: 
#   '1_GUIDANCE_SCALE_SWEEP'         : Quét Scale s ∈ [7.5, 12.5, 15.0, 17.5, 20.0] (~15-20 phút nếu tái sử dụng Phase 1)
#   '2_PARTICLE_SCALING'             : Quét số hạt N ∈ [3, 5, 10, 20, 50, 100]
#   '3_PHASE1_LOOKAHEAD_STEPS_SWEEP' : Quét số bước DPM Phase 1 K ∈ [2, 3, 5, 8] (Chứng minh RS kháng sai số Solver)
#   'ALL'                            : Chạy tuần tự cả 3 thực nghiệm
EXPERIMENT_MODE = '1_GUIDANCE_SCALE_SWEEP'

# 🎯 QUY MÔ PROMPT: 
#   - Mặc định NUM_PROMPTS = 20 để chạy nhanh (15-30 phút), kiểm chứng quy luật khoa học & vẽ đồ thị.
#   - Để chạy Full Benchmark chính thức của Bảng 2, đổi NUM_PROMPTS = 553.
NUM_PROMPTS = 20

REUSE_EXISTING_PHASE1 = True          # Tự động tái sử dụng Phase 1 latents nếu đã có sẵn
DEFAULT_LOOKAHEAD_STEPS = 5           # Số bước DPM lookahead mặc định (khi không chạy Exp 3)
LOOKAHEAD_SEED = 100                  # Seed sinh hạt lookahead

# Cấu hình dải tham số cho từng thực nghiệm
GUIDANCE_SCALES_SWEEP       = [7.5, 12.5, 15.0, 17.5, 20.0]    # Dải Scale cho Exp 1
PARTICLE_COUNTS_SWEEP       = [3, 5, 10, 20, 50, 100]          # Dải hạt cho Exp 2
LOOKAHEAD_STEPS_SWEEP       = [2, 3, 5, 8]                     # Dải bước DPM Phase 1 cho Exp 3

# Tham số chuẩn của RS-LiDAR
SIGMA = 1.0                           # Độ lệch chuẩn Randomized Smoothing cho RS-LiDAR
NUM_MC_SAMPLES = 4                    # Số mẫu Monte Carlo tính E[R(x+xi)]
LAMBDA = 5000                         # Hệ số Softmax reward chuẩn bài báo
# ==============================================================================


## 3. Đồng Bộ Repository & Cài Đặt Thư Viện Cần Thiết
Tự động clone code mới nhất từ GitHub và cài đặt môi trường tối giản, không xung đột dependency.

In [ ]:
import os, shutil, glob, json, random, subprocess, threading, time
import pandas as pd, numpy as np, matplotlib.pyplot as plt

# 1. Đồng bộ mã nguồn
REPO_DIR = '/kaggle/working/RS-LiDAR'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/leekwanreal/RS-LiDAR.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull origin main

WORKDIR = f"{REPO_DIR}/Diffusion-LiDAR-Sampling" if os.path.exists(f"{REPO_DIR}/Diffusion-LiDAR-Sampling") else REPO_DIR
os.chdir(WORKDIR)
%cd {WORKDIR}
print("📂 Thư mục làm việc:", os.getcwd())

# 2. Cài đặt các thư viện phụ thuộc chuẩn
!pip install -q --upgrade protobuf
!pip install -q transformers==4.38.2 diffusers==0.31.0 accelerate==1.2.1 safetensors huggingface-hub einops ftfy timm peft
!pip install -q git+https://github.com/THUDM/ImageReward.git
!pip install -q scipy matplotlib seaborn pandas tabulate open_clip_torch

# 3. Hàm hỗ trợ chạy song song GPU
def run_parallel_or_serial(cmd0, cmd1=None):
    n_g = torch.cuda.device_count() if torch.cuda.is_available() else 1
    if n_g >= 2 and cmd1 is not None:
        p0 = subprocess.Popen(cmd0, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        p1 = subprocess.Popen(cmd1, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        def stream(p, pfx):
            for line in iter(p.stdout.readline, ''):
                if line.strip(): print(f"{pfx} {line.strip()}")
            p.stdout.close()
        t0 = threading.Thread(target=stream, args=(p0, '[GPU 0]')); t1 = threading.Thread(target=stream, args=(p1, '[GPU 1]'))
        t0.start(); t1.start(); t0.join(); t1.join()
        return (p0.wait() == 0 and p1.wait() == 0)
    else:
        res = subprocess.run(cmd0, shell=True)
        return (res.returncode == 0)

print("✅ Môi trường đã sẵn sàng!")


## 4. Chuẩn Bị Tập Prompt Thực Nghiệm (Tái Lập Đảm Bảo 100% Seed 42)

In [ ]:
PROMPT_FILE = 'prompt_files/geneval_metadata.jsonl'
EXP_PROMPT_FILE = '/kaggle/working/exp_prompts_{NUM_PROMPTS}.jsonl'

all_prompts = []
with open(PROMPT_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip(): all_prompts.append(json.loads(line.strip()))

random.seed(42)
if NUM_PROMPTS >= len(all_prompts):
    selected_prompts = all_prompts
    print(f"🌟 CHẠY TOÀN BỘ BENCHMARK BẢNG 2: {len(selected_prompts)} prompts!")
else:
    selected_prompts = random.sample(all_prompts, NUM_PROMPTS)
    print(f"🔬 CHẠY ABLATION STUDY ĐẠI DIỆN: {len(selected_prompts)} prompts (Seed 42)!")

with open(EXP_PROMPT_FILE, 'w', encoding='utf-8') as f:
    for item in selected_prompts:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print(f"✅ Đã lưu tập prompt vào: {EXP_PROMPT_FILE}")


## 5. [PHASE 1] Chuẩn Bị & Tái Sử Dụng Dữ Liệu Hạt Lookahead
Tự động chuẩn bị tập hạt cho cả Vanilla (điểm thưởng gốc) và RS-LiDAR (điểm thưởng smoothed $R_\sigma$). Nhờ cơ chế `--reuse_latents_from`, RS-LiDAR chỉ cần chấm điểm Monte Carlo mà không phải sinh lại từ đầu, tiết kiệm 95% thời gian!

In [ ]:
LOOKAHEAD_BASE_DIR = 'Lookahead_samples'
os.makedirs(LOOKAHEAD_BASE_DIR, exist_ok=True)
n_g = torch.cuda.device_count() if torch.cuda.is_available() else 1

# Xác định danh sách các cặp (Số Hạt N, Số Bước DPM K) cần chuẩn bị cho Phase 1
configs_p1 = set()
if EXPERIMENT_MODE in ['1_GUIDANCE_SCALE_SWEEP', 'ALL']:
    configs_p1.add((50, DEFAULT_LOOKAHEAD_STEPS))
if EXPERIMENT_MODE in ['2_PARTICLE_SCALING', 'ALL']:
    for n_p in PARTICLE_COUNTS_SWEEP:
        configs_p1.add((n_p, DEFAULT_LOOKAHEAD_STEPS))
if EXPERIMENT_MODE in ['3_PHASE1_LOOKAHEAD_STEPS_SWEEP', 'ALL']:
    for k_s in LOOKAHEAD_STEPS_SWEEP:
        configs_p1.add((50, k_s))

print(f"📋 Danh sách các cấu hình Phase 1 cần chuẩn bị: {sorted(list(configs_p1))}")

for (n_p, k_s) in sorted(list(configs_p1)):
    vanilla_tag = f"100_{n_p}_{k_s}"
    rs_tag = f"RS_100_{n_p}_{k_s}_sig{SIGMA}_M{NUM_MC_SAMPLES}"
    
    v_dir = os.path.join(LOOKAHEAD_BASE_DIR, vanilla_tag)
    rs_dir = os.path.join(LOOKAHEAD_BASE_DIR, rs_tag)
    
    # 1. Kiểm tra / Sinh Vanilla Lookahead
    v_done = len(glob.glob(f"{v_dir}/*/results.json")) >= NUM_PROMPTS
    if v_done and REUSE_EXISTING_PHASE1:
        print(f"  ⏩ Vanilla Lookahead (N={n_p}, Steps={k_s}) đã có sẵn tại {v_dir}. Bỏ qua.")
    else:
        print(f"  🚀 Sinh Vanilla Lookahead (N={n_p}, Steps={k_s})...")
        if n_g >= 2:
            cmd0 = f"python -u lookahead_sampling.py --prompt_path '{EXP_PROMPT_FILE}' --model_name 'runwayml/stable-diffusion-v1-5' --num_particles {n_p} --num_inference_steps {k_s} --seed {LOOKAHEAD_SEED} --run_name '{vanilla_tag}' --save_individual_images True --num_shards 2 --shard_id 0 --gpu_id 0 --resume"
            cmd1 = f"python -u lookahead_sampling.py --prompt_path '{EXP_PROMPT_FILE}' --model_name 'runwayml/stable-diffusion-v1-5' --num_particles {n_p} --num_inference_steps {k_s} --seed {LOOKAHEAD_SEED} --run_name '{vanilla_tag}' --save_individual_images True --num_shards 2 --shard_id 1 --gpu_id 1 --resume"
            run_parallel_or_serial(cmd0, cmd1)
        else:
            cmd = f"python -u lookahead_sampling.py --prompt_path '{EXP_PROMPT_FILE}' --model_name 'runwayml/stable-diffusion-v1-5' --num_particles {n_p} --num_inference_steps {k_s} --seed {LOOKAHEAD_SEED} --run_name '{vanilla_tag}' --save_individual_images True --resume"
            run_parallel_or_serial(cmd)
            
    # 2. Kiểm tra / Chấm điểm RS-LiDAR Lookahead (Tái sử dụng latents từ Vanilla)
    rs_done = len(glob.glob(f"{rs_dir}/*/results.json")) >= NUM_PROMPTS
    if rs_done and REUSE_EXISTING_PHASE1:
        print(f"  ⏩ RS-LiDAR Lookahead (N={n_p}, Steps={k_s}) đã có sẵn tại {rs_dir}. Bỏ qua.")
    else:
        print(f"  ⚡ Chấm điểm RS-LiDAR Monte Carlo (N={n_p}, Steps={k_s}, sigma={SIGMA}, M={NUM_MC_SAMPLES}) tái sử dụng latents...")
        if n_g >= 2:
            cmd0 = f"python -u lookahead_sampling.py --prompt_path '{EXP_PROMPT_FILE}' --model_name 'runwayml/stable-diffusion-v1-5' --num_particles {n_p} --num_inference_steps {k_s} --seed {LOOKAHEAD_SEED} --run_name '{rs_tag}' --use_smoothing --sigma {SIGMA} --num_mc_samples {NUM_MC_SAMPLES} --reuse_latents_from '{vanilla_tag}' --num_shards 2 --shard_id 0 --gpu_id 0 --resume"
            cmd1 = f"python -u lookahead_sampling.py --prompt_path '{EXP_PROMPT_FILE}' --model_name 'runwayml/stable-diffusion-v1-5' --num_particles {n_p} --num_inference_steps {k_s} --seed {LOOKAHEAD_SEED} --run_name '{rs_tag}' --use_smoothing --sigma {SIGMA} --num_mc_samples {NUM_MC_SAMPLES} --reuse_latents_from '{vanilla_tag}' --num_shards 2 --shard_id 1 --gpu_id 1 --resume"
            run_parallel_or_serial(cmd0, cmd1)
        else:
            cmd = f"python -u lookahead_sampling.py --prompt_path '{EXP_PROMPT_FILE}' --model_name 'runwayml/stable-diffusion-v1-5' --num_particles {n_p} --num_inference_steps {k_s} --seed {LOOKAHEAD_SEED} --run_name '{rs_tag}' --use_smoothing --sigma {SIGMA} --num_mc_samples {NUM_MC_SAMPLES} --reuse_latents_from '{vanilla_tag}' --resume"
            run_parallel_or_serial(cmd)

print("✅ Hoàn tất chuẩn bị dữ liệu Phase 1 cho cả Vanilla và RS-LiDAR!")


## 6. [PHASE 2] Thực Thi Target Sampling Toàn Diện Cho Dải Khảo Sát
Tự động chạy song song 2 GPU trên Kaggle hoặc đơn GPU trên Colab cho tất cả các cấu hình của thí nghiệm đã chọn.

In [ ]:
TARGET_OUTPUT_ROOT = 'Target_samples'
os.makedirs(TARGET_OUTPUT_ROOT, exist_ok=True)
n_g = torch.cuda.device_count() if torch.cuda.is_available() else 1

runs_to_execute = []

# 1. Chế độ Thực Nghiệm 1: Guidance Scale Sweep
if EXPERIMENT_MODE in ['1_GUIDANCE_SCALE_SWEEP', 'ALL']:
    print(f"📋 Lên lịch Thực Nghiệm 1 (Guidance Scale Stress-Test) với s ∈ {GUIDANCE_SCALES_SWEEP}...")
    for s_val in GUIDANCE_SCALES_SWEEP:
        runs_to_execute.append({
            'exp': 'Exp1_Scale',
            'method': 'Vanilla_LiDAR',
            'scale': s_val,
            'n_particles': 50,
            'p1_steps': DEFAULT_LOOKAHEAD_STEPS,
            'lookahead_tag': f"100_50_{DEFAULT_LOOKAHEAD_STEPS}",
            'run_name': f"LiDAR_Exp1_s{s_val}_n50_k{DEFAULT_LOOKAHEAD_STEPS}"
        })
        runs_to_execute.append({
            'exp': 'Exp1_Scale',
            'method': 'RS_LiDAR',
            'scale': s_val,
            'n_particles': 50,
            'p1_steps': DEFAULT_LOOKAHEAD_STEPS,
            'lookahead_tag': f"RS_100_50_{DEFAULT_LOOKAHEAD_STEPS}_sig{SIGMA}_M{NUM_MC_SAMPLES}",
            'run_name': f"RSLiDAR_Exp1_s{s_val}_n50_k{DEFAULT_LOOKAHEAD_STEPS}"
        })

# 2. Chế độ Thực Nghiệm 2: Particle Scaling & Ultra-Low Budget
if EXPERIMENT_MODE in ['2_PARTICLE_SCALING', 'ALL']:
    print(f"📋 Lên lịch Thực Nghiệm 2 (Particle Scaling & Ultra-Low Budget) với N ∈ {PARTICLE_COUNTS_SWEEP}...")
    for n_val in PARTICLE_COUNTS_SWEEP:
        runs_to_execute.append({
            'exp': 'Exp2_Particle',
            'method': 'Vanilla_LiDAR',
            'scale': 12.5,
            'n_particles': n_val,
            'p1_steps': DEFAULT_LOOKAHEAD_STEPS,
            'lookahead_tag': f"100_{n_val}_{DEFAULT_LOOKAHEAD_STEPS}",
            'run_name': f"LiDAR_Exp2_s12.5_n{n_val}_k{DEFAULT_LOOKAHEAD_STEPS}"
        })
        runs_to_execute.append({
            'exp': 'Exp2_Particle',
            'method': 'RS_LiDAR',
            'scale': 12.5,
            'n_particles': n_val,
            'p1_steps': DEFAULT_LOOKAHEAD_STEPS,
            'lookahead_tag': f"RS_100_{n_val}_{DEFAULT_LOOKAHEAD_STEPS}_sig{SIGMA}_M{NUM_MC_SAMPLES}",
            'run_name': f"RSLiDAR_Exp2_s12.5_n{n_val}_k{DEFAULT_LOOKAHEAD_STEPS}"
        })

# 3. Chế độ Thực Nghiệm 3: Phase 1 Lookahead Horizon (Solver Truncation Error)
if EXPERIMENT_MODE in ['3_PHASE1_LOOKAHEAD_STEPS_SWEEP', 'ALL']:
    print(f"📋 Lên lịch Thực Nghiệm 3 (Phase 1 Lookahead Horizon) với K ∈ {LOOKAHEAD_STEPS_SWEEP}...")
    for k_val in LOOKAHEAD_STEPS_SWEEP:
        runs_to_execute.append({
            'exp': 'Exp3_Phase1Steps',
            'method': 'Vanilla_LiDAR',
            'scale': 12.5,
            'n_particles': 50,
            'p1_steps': k_val,
            'lookahead_tag': f"100_50_{k_val}",
            'run_name': f"LiDAR_Exp3_s12.5_n50_k{k_val}"
        })
        runs_to_execute.append({
            'exp': 'Exp3_Phase1Steps',
            'method': 'RS_LiDAR',
            'scale': 12.5,
            'n_particles': 50,
            'p1_steps': k_val,
            'lookahead_tag': f"RS_100_50_{k_val}_sig{SIGMA}_M{NUM_MC_SAMPLES}",
            'run_name': f"RSLiDAR_Exp3_s12.5_n50_k{k_val}"
        })

print(f"\n🚀 TỔNG SỐ LƯỢT CHẠY TARGET SAMPLING: {len(runs_to_execute)} runs")

for idx_run, run_cfg in enumerate(runs_to_execute):
    r_name = run_cfg['run_name']
    s_val = run_cfg['scale']
    n_parts = run_cfg['n_particles']
    lk_tag = run_cfg['lookahead_tag']
    k_steps = run_cfg['p1_steps']
    
    print("\n" + "="*90)
    print(f"[{idx_run+1}/{len(runs_to_execute)}] ĐANG CHẠY: {r_name}")
    print(f"  • Thuật toán: {run_cfg['method']} | Scale s: {s_val} | Particles N: {n_parts} | Phase 1 DPM: {k_steps} steps")
    print(f"  • Lookahead nguồn: {lk_tag}")
    print("="*90)
    
    if n_g >= 2:
        cmd0 = f"python -u LiDAR_sampling.py --prompt_path '{EXP_PROMPT_FILE}' --lookahead_path '{lk_tag}' --scale {s_val} --top_k {n_parts} --lmbda {LAMBDA} --num_inference_steps 50 --eta 0.0 --use_rag --resample_t_end 200 --run_name '{r_name}' --save_individual_images --num_shards 2 --shard_id 0 --gpu_id 0 --resume"
        cmd1 = f"python -u LiDAR_sampling.py --prompt_path '{EXP_PROMPT_FILE}' --lookahead_path '{lk_tag}' --scale {s_val} --top_k {n_parts} --lmbda {LAMBDA} --num_inference_steps 50 --eta 0.0 --use_rag --resample_t_end 200 --run_name '{r_name}' --save_individual_images --num_shards 2 --shard_id 1 --gpu_id 1 --resume"
        run_parallel_or_serial(cmd0, cmd1)
    else:
        cmd = f"python -u LiDAR_sampling.py --prompt_path '{EXP_PROMPT_FILE}' --lookahead_path '{lk_tag}' --scale {s_val} --top_k {n_parts} --lmbda {LAMBDA} --num_inference_steps 50 --eta 0.0 --use_rag --resample_t_end 200 --run_name '{r_name}' --save_individual_images --resume"
        run_parallel_or_serial(cmd)

print("\n🎉 HOÀN TẤT TOÀN BỘ CÁC LƯỢT CHẠY TARGET SAMPLING!")


## 7. Báo Cáo Kết Quả, Bảng Tổng Hợp & Đồ Thị Động Học Khoa Học
Tự động đọc toàn bộ kết quả, in bảng so sánh, vẽ đồ thị phản ánh ranh giới sụp đổ của LiDAR vs sức bền của RS-LiDAR trên cả 3 khía cạnh: Guidance Scale, Particle Scaling và Phase 1 Lookahead Horizon.

In [ ]:
import glob, json, os, numpy as np, pandas as pd, matplotlib.pyplot as plt

# Tổng hợp kết quả từ các run trong Target_samples
summary_rows = []
target_runs = glob.glob(f"{TARGET_OUTPUT_ROOT}/*")

for r_dir in sorted(target_runs):
    r_name = os.path.basename(r_dir)
    if not os.path.isdir(r_dir): continue
    
    metrics_f = os.path.join(r_dir, 'final_metrics.json')
    ir_val, clip_val, hps_val = np.nan, np.nan, np.nan
    
    # 1. Đọc từ final_metrics.json
    if os.path.exists(metrics_f):
        try:
            with open(metrics_f, 'r') as f:
                m_data = json.load(f)
                ir_val = m_data.get('ImageReward', {}).get('mean', np.nan)
                clip_val = m_data.get('Clip-Score', {}).get('mean', np.nan)
                hps_val = m_data.get('HumanPreference', {}).get('mean', np.nan)
        except Exception:
            pass
    
    # 2. Dự phòng: Tính trung bình từ kết quả các prompts riêng rẽ
    if np.isnan(ir_val):
        p_results = glob.glob(f"{r_dir}/*/results.json")
        ir_list, clip_list = [], []
        for pr_f in p_results:
            try:
                with open(pr_f, 'r') as f:
                    data = json.load(f)
                    if 'ImageReward' in data: ir_list.append(data['ImageReward']['mean'])
                    if 'Clip-Score' in data: clip_list.append(data['Clip-Score']['mean'])
            except Exception:
                pass
        if ir_list: ir_val = float(np.mean(ir_list))
        if clip_list: clip_val = float(np.mean(clip_list))
        
    # Phân loại phương pháp và tham số từ run_name
    method_type = 'RS_LiDAR' if 'RSLiDAR' in r_name else 'Vanilla_LiDAR'
    
    # Trích xuất Scale
    scale_val = 12.5
    for s_v in [7.5, 12.5, 15.0, 17.5, 20.0]:
        if f"_s{s_v}_" in r_name or f"_s{s_v}" in r_name:
            scale_val = s_v; break
            
    # Trích xuất Particles
    n_part_val = 50
    for n_v in [3, 5, 10, 20, 50, 100]:
        if f"_n{n_v}_" in r_name or f"_n{n_v}" in r_name:
            n_part_val = n_v; break
            
    # Trích xuất Phase 1 Lookahead Steps (k)
    k_steps_val = 5
    for k_v in [2, 3, 5, 8]:
        if f"_k{k_v}" in r_name:
            k_steps_val = k_v; break
            
    # Phân loại nhóm thí nghiệm
    exp_group = 'Unknown'
    if 'Exp1' in r_name or ('_s' in r_name and '_n50' in r_name and k_steps_val == 5):
        exp_group = 'Exp1_Scale'
    elif 'Exp2' in r_name or ('_n' in r_name and scale_val == 12.5 and k_steps_val == 5):
        exp_group = 'Exp2_Particle'
    elif 'Exp3' in r_name or ('_k' in r_name and scale_val == 12.5 and n_part_val == 50):
        exp_group = 'Exp3_Phase1Steps'
            
    summary_rows.append({
        'Run_Name': r_name,
        'Exp_Group': exp_group,
        'Method': method_type,
        'Guidance_Scale': scale_val,
        'Particles_N': n_part_val,
        'Phase1_DPM_Steps': k_steps_val,
        'ImageReward': round(ir_val, 4) if not np.isnan(ir_val) else 'N/A',
        'CLIP_Score': round(clip_val, 4) if not np.isnan(clip_val) else 'N/A',
        'HPS_v2.1': round(hps_val, 4) if not np.isnan(hps_val) else 'N/A'
    })

df_summary = pd.DataFrame(summary_rows)
print("="*110)
print("📊 BẢNG TỔNG HỢP TOÀN BỘ KẾT QUẢ THỰC NGHIỆM ĐỀ XUẤT:")
print("="*110)
from IPython.display import display
display(df_summary)

# Lưu CSV
csv_out_path = '/kaggle/working/advanced_experiments_summary.csv' if is_kaggle else '/content/advanced_experiments_summary.csv'
df_summary.to_csv(csv_out_path, index=False)
print(f"💾 Đã lưu bảng tổng hợp tại: {csv_out_path}")

# -------------------------------------------------------------------------
# VẼ ĐỒ THỊ KHOA HỌC CHO TỪNG THỰC NGHIỆM
# -------------------------------------------------------------------------
# Đồ thị 1: Guidance Scale Curve (Exp 1)
df_exp1 = df_summary[df_summary['Exp_Group'] == 'Exp1_Scale'].copy()
if len(df_exp1) >= 2:
    try:
        plt.figure(figsize=(9, 5.5))
        for m_name, color, marker, ls in [('Vanilla_LiDAR', 'red', 'o', '--'), ('RS_LiDAR', 'green', 's', '-')]:
            sub = df_exp1[df_exp1['Method'] == m_name].sort_values('Guidance_Scale')
            valid = sub[sub['ImageReward'] != 'N/A']
            if len(valid) > 0:
                plt.plot(valid['Guidance_Scale'], valid['ImageReward'].astype(float), label=m_name, color=color, marker=marker, linestyle=ls, linewidth=2.5)
        plt.title("Exp 1: Guidance Scale Stress-Test (Lipschitz Stability Margin)", fontsize=12, fontweight='bold')
        plt.xlabel("Guidance Scale s", fontsize=11); plt.ylabel("ImageReward ↑", fontsize=11)
        plt.axvline(x=15.0, color='gray', linestyle=':', label='LiDAR Peak Threshold (s=15.0)')
        plt.grid(True, linestyle='--', alpha=0.5); plt.legend(fontsize=10); plt.tight_layout()
        p1_plot_f = '/kaggle/working/exp1_guidance_scale_curve.png' if is_kaggle else '/content/exp1_guidance_scale_curve.png'
        plt.savefig(p1_plot_f, dpi=200); plt.show()
        print(f"📈 Đã lưu đồ thị Exp 1 tại: {p1_plot_f}")
    except Exception as e: print(f"Vẽ đồ thị Exp 1 lỗi: {e}")

# Đồ thị 2: Particle Scaling & Ultra-Low Budget Curve (Exp 2)
df_exp2 = df_summary[df_summary['Exp_Group'] == 'Exp2_Particle'].copy()
if len(df_exp2) >= 2:
    try:
        plt.figure(figsize=(9, 5.5))
        for m_name, color, marker, ls in [('Vanilla_LiDAR', 'red', 'o', '--'), ('RS_LiDAR', 'green', 's', '-')]:
            sub = df_exp2[df_exp2['Method'] == m_name].sort_values('Particles_N')
            valid = sub[sub['ImageReward'] != 'N/A']
            if len(valid) > 0:
                plt.plot(valid['Particles_N'], valid['ImageReward'].astype(float), label=m_name, color=color, marker=marker, linestyle=ls, linewidth=2.5)
        plt.title("Exp 2: Particle Scaling & Ultra-Low Budget Consensus", fontsize=12, fontweight='bold')
        plt.xlabel("Lookahead Particles N", fontsize=11); plt.ylabel("ImageReward ↑", fontsize=11)
        plt.grid(True, linestyle='--', alpha=0.5); plt.legend(fontsize=10); plt.tight_layout()
        p2_plot_f = '/kaggle/working/exp2_particle_scaling_curve.png' if is_kaggle else '/content/exp2_particle_scaling_curve.png'
        plt.savefig(p2_plot_f, dpi=200); plt.show()
        print(f"📈 Đã lưu đồ thị Exp 2 tại: {p2_plot_f}")
    except Exception as e: print(f"Vẽ đồ thị Exp 2 lỗi: {e}")

# Đồ thị 3: Phase 1 Lookahead Horizon Curve (Exp 3)
df_exp3 = df_summary[df_summary['Exp_Group'] == 'Exp3_Phase1Steps'].copy()
if len(df_exp3) >= 2:
    try:
        plt.figure(figsize=(9, 5.5))
        for m_name, color, marker, ls in [('Vanilla_LiDAR', 'red', 'o', '--'), ('RS_LiDAR', 'green', 's', '-')]:
            sub = df_exp3[df_exp3['Method'] == m_name].sort_values('Phase1_DPM_Steps')
            valid = sub[sub['ImageReward'] != 'N/A']
            if len(valid) > 0:
                plt.plot(valid['Phase1_DPM_Steps'], valid['ImageReward'].astype(float), label=m_name, color=color, marker=marker, linestyle=ls, linewidth=2.5)
        plt.title("Exp 3: Phase 1 Lookahead Horizon (DPM Steps vs Solver Error Damping)", fontsize=12, fontweight='bold')
        plt.xlabel("Phase 1 DPM Lookahead Steps K", fontsize=11); plt.ylabel("ImageReward ↑", fontsize=11)
        plt.grid(True, linestyle='--', alpha=0.5); plt.legend(fontsize=10); plt.tight_layout()
        p3_plot_f = '/kaggle/working/exp3_phase1_steps_curve.png' if is_kaggle else '/content/exp3_phase1_steps_curve.png'
        plt.savefig(p3_plot_f, dpi=200); plt.show()
        print(f"📈 Đã lưu đồ thị Exp 3 tại: {p3_plot_f}")
    except Exception as e: print(f"Vẽ đồ thị Exp 3 lỗi: {e}")


## 8. Đóng Gói Toàn Bộ Kết Quả & Ảnh Độc Quyền (1-Click Download)
Nén toàn bộ bảng CSV, đồ thị biểu diễn và các ảnh đã sinh ra thành file zip để tải về chỉ với 1 cú click.

In [ ]:
zip_path = '/kaggle/working/rs_lidar_advanced_results.zip'
print(f"📦 Đang nén toàn bộ kết quả và ảnh thực nghiệm vào: {zip_path}...")
!zip -r -q {zip_path} {TARGET_OUTPUT_ROOT} /kaggle/working/advanced_experiments_summary.csv
if os.path.exists(zip_path):
    size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"\n✅ HOÀN TẤT ĐÓNG GÓI! Kích thước file: {size_mb:.2f} MB")
    print("Tải file zip trực tiếp từ tab Output bên phải Kaggle để xem toàn bộ ảnh và bảng số liệu!")
